# NUTDTS 816 Time Series Analysis
## L07 ARIMA: Box-Jenkins identification, estimation, diagnostics

Lab notebook for Chapter 4 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 4.3 Worked example I: a stationary series (US consumption growth)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import tsdata
usc = tsdata.uschange()['Consumption']; usc.name = 'consumption growth (%)'
fig, axes = plt.subplots(1, 3, figsize=(11, 3), gridspec_kw={'width_ratios': [2, 1, 1]})
usc.plot(ax=axes[0], lw=0.9, title='Quarterly US consumption growth (%)'); axes[0].set_xlabel('')
plot_acf(usc, lags=24, ax=axes[1], title='ACF'); plot_pacf(usc, lags=24, ax=axes[2], title='PACF')
for ax in axes[1:]: ax.set_ylim(-0.5, 1)
_caption = 'A stationary series. The ACF has three significant early lags and tails off; the PACF has significant lags 1 to 3. Candidates: AR(3), MA(3), or a mixed model.'

In [ ]:
cands = [(3, 0, 0), (0, 0, 3), (1, 0, 1), (1, 0, 2), (2, 0, 1), (1, 0, 3), (2, 0, 2)]
rows = []
for o in cands:
    f = ARIMA(usc, order=o).fit()
    lb = acorr_ljungbox(f.resid, lags=[8], model_df=o[0] + o[2], return_df=True)
    rows.append({'order': o, 'AIC': round(f.aic, 2), 'AICc': round(f.aicc, 2), 'BIC': round(f.bic, 2),
                 'LB(8) p': round(lb.lb_pvalue.iloc[0], 3), 'sigma2': round(f.params['sigma2'], 4)})
print(pd.DataFrame(rows).sort_values('AICc').to_string(index=False))

### 4.5 Diagnostic checking

In [ ]:
fit = ARIMA(usc, order=(3, 0, 0)).fit()
print(fit.summary().tables[1])
fig = fit.plot_diagnostics(figsize=(10, 6))
_caption = 'statsmodels diagnostic panel for the AR(3): standardised residuals, their histogram against N(0,1), Q-Q plot, and residual correlogram.'

In [ ]:
lb = acorr_ljungbox(fit.resid, lags=[4, 8, 12, 16], model_df=3, return_df=True)
print(lb.round(3).to_string())
print('\nAR root modulus:', np.round(np.abs(fit.arroots), 3), '(all should exceed 1 for stationarity)')

## Exercises

1. Write the ARIMA(1,1,1) model with drift in full (no backshift notation) and derive its one- and two-step forecast equations.
2. For the AR(3) fitted to consumption growth, compute $\psi_1 = \phi_1$, $\psi_2 = \phi_1^2 + \phi_2$ and $\psi_3$ from the estimated parameters and verify that the reported standard error at horizon 2 equals $\hat\sigma\sqrt{1 + \psi_1^2}$.
3. Fit ARIMA models to the `elecequip` series (monthly electrical equipment orders index) after seasonal adjustment with STL: choose $d$, shortlist candidates, compare by AICc, diagnose, and forecast 24 months with intervals. Compare with the drift method on the last 24 months held out.

In [ ]:
# Your work here
